
# DOPe-Bench v3: Better Experiment Pipeline

Builds on v2 (OpenRouter models + MLBT/AC/CRG task suite). This version adds the
five things a benchmark paper needs to survive ICRA review that v2 didn't have:

1. **Explicit evaluation strata** (Sec. 3) - Standard, Difficulty-stratified (L1/L2/L3),
   and Archetype-support-stratified - instead of a single flat test set.
2. **Non-generative baselines** (Sec. 5) - random and frequency baselines, so every
   VLM number in the paper is contextualized against a trivial floor. A reviewer's
   first question for any recognition benchmark is "how much of this is beatable by
   just predicting the most common tags?" - this answers it directly.
3. **Statistical significance testing** (Sec. 15) - bootstrap confidence intervals on
   every F1 number, and paired bootstrap tests between models, so the main results
   table reports which differences are actually distinguishable from noise rather
   than presenting a leaderboard of point estimates.
4. **Judge validation against human ratings** (Sec. 14) - a template + correlation
   pipeline so the CRG judge score is reported with a measured human-agreement
   number instead of being asserted as reliable.
5. **A reproducibility manifest** (Sec. 19) - every run is logged with prompt
   hashes, model snapshot identifiers, seed, and timestamp, because OpenRouter
   models silently get deprecated/replaced (this happened to `gemini-3-pro-preview`
   mid-cycle, see Sec. 0) and a benchmark paper needs to be able to say exactly
   which model version produced which number.

Everything from v2 that isn't listed above is unchanged: same OpenRouter model
registry, same MLBT→AC→CRG task chain with oracle/pipeline conditions, same
Compose-lite/Early-lite proxies, same CSV-only dataset-analysis tables, same
honest limitations around Table I / DOPe-Risk / DOPe-Intervene / BTR / CIR. This
notebook is CPU-only throughout (all inference via OpenRouter); the local-Qwen3-VL
Modal appendix remains optional.

## Fig. DOPe-Bench v3 pipeline

```text
                    Scenario clip (frames)
                              │
              ┌───────────────┴───────────────┐
              │      Evaluation strata          │
              │  Standard · L1/L2/L3 · Compose  │
              │  · Archetype-support            │
              └───────────────┬───────────────┘
                              ▼
        Baselines (random, frequency)   Task 1 — MLBT
                    │                    frames → behavior tags
                    │                            │ pred tags
                    │                            ▼
                    │                    Task 2 — AC (oracle / pipeline)
                    │                    frames + tags → archetype(s)
                    │                            │ pred archetypes
                    │                            ▼
                    │                    Task 3 — CRG (oracle / pipeline)
                    │                    frames + tags + archetypes →
                    │                    7-axis rationale
                    │                            │
                    │                ┌───────────┴───────────┐
                    │                ▼                       ▼
                    │        Grounding-precision     LLM-judge CLAIR-style
                    │        (rule-based, per axis)  score, validated against
                    │                                a human-rating subsample
                    ▼                                        │
        ┌──────────────────────────────────────────────────┐│
        │  Statistical layer: bootstrap CIs on every F1,    ││
        │  paired significance tests between models,        ││
        │  judge-vs-human correlation (Spearman)             │
        └──────────────────────────────────────────────────┘
                              │
                              ▼
                Reproducibility manifest (prompt hash,
                model snapshot, seed, timestamp)
```


## 0. VLMs to test via OpenRouter

Unchanged from v2 — current as of September 2026. Prices per 1M tokens; calibrate
image-token cost against Sec. 18's estimator after a small pilot.

| Tier | Model | `model_id` | Context | Input $/M | Output $/M |
|---|---|---|---|---|---|
| Frontier | GPT-5.6 Sol | `openai/gpt-5.6-sol` | 1.05M | $2.00 | $10.00 |
| Frontier | Claude Opus 5 | `anthropic/claude-opus-5` | 1M | $5.00 | $25.00 |
| Frontier | Gemini 3.1 Pro Preview | `google/gemini-3.1-pro-preview` | 1.05M | $2.00 | $12.00 |
| Efficient | GPT-5.6 Luna | `openai/gpt-5.6-luna` | 1.05M | $0.20 | $1.20 |
| Efficient | Claude Sonnet 5 | `anthropic/claude-sonnet-5` | 1M | $2.00 | $10.00 |
| Efficient | Gemini 3.7 Flash | `google/gemini-3.7-flash` | 1.05M | $0.75 | $3.75 |
| Open-weight, large | Qwen3-VL-32B-Instruct | `qwen/qwen3-vl-32b-instruct` | 131K | $0.104 | $0.416 |
| Open-weight, large | Qwen3-VL-235B-A22B-Instruct | `qwen/qwen3-vl-235b-a22b-instruct` | 262K | $0.20 | $0.88 |
| Open-weight, large | GLM-4.6V | `z-ai/glm-4.6v` | 131K | $0.30 | $0.90 |
| Open-weight, small | Qwen3-VL-8B-Instruct | `qwen/qwen3-vl-8b-instruct` | 256K | ~$0.05 | ~$0.10 |
| Reasoning-tuned (CRG only) | Qwen3-VL-30B-A3B-Thinking | `qwen/qwen3-vl-30b-a3b-thinking` | 131K | mid | mid |
| Budget / free | MiniMax M3 | `minimax/minimax-m3:free` | 1.05M | $0 | $0 |

**Reproducibility note (new in v3):** `gemini-3-pro-preview` was deprecated by
Google mid-cycle in favor of `gemini-3.1-pro-preview` while this benchmark was
being built — a concrete example of why Sec. 19's manifest records the exact
model ID string and run timestamp for every result you report, and why the paper
should state the evaluation window (e.g. "all API results collected between
DATE and DATE") rather than implying a fixed, permanently reproducible artifact.
Judge model: `claude-opus-5` by default (never the same model as a subject under
test — see Sec. 0 of v2 for the full rationale).


## 1. Environment setup (CPU-only)

In [ ]:

!pip install -q pandas numpy scipy scikit-learn matplotlib seaborn rapidfuzz tqdm
!pip install -q yt-dlp opencv-python-headless
!pip install -q openai


In [ ]:

import os, re, json, time, random, base64, subprocess, hashlib, platform, datetime
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
from scipy import stats as sstats
from rapidfuzz import fuzz, process as rf_process
from tqdm.auto import tqdm
from openai import OpenAI

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_CSV    = Path("1-video_annotations_dataset.csv")
VIDEO_DIR   = Path("data/videos")
FRAME_DIR   = Path("data/frames")
RESULTS_DIR = Path("results")
for d in (VIDEO_DIR, FRAME_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

MIN_FRAMES = 4
SAMPLE_FPS = 2.0

def num_frames_for_scenario(start_frame: int, end_frame: int, source_fps: float) -> int:
    duration_s = (end_frame - start_frame) / source_fps
    return max(MIN_FRAMES, round(duration_s * SAMPLE_FPS))

MODEL_REGISTRY = {
    "gpt-5.6-sol":            {"id": "openai/gpt-5.6-sol",              "cat": "A", "price_in": 2.00,  "price_out": 10.00},
    "claude-opus-5":          {"id": "anthropic/claude-opus-5",         "cat": "A", "price_in": 5.00,  "price_out": 25.00},
    "gemini-3.1-pro-preview": {"id": "google/gemini-3.1-pro-preview",   "cat": "A", "price_in": 2.00,  "price_out": 12.00},
    "gpt-5.6-luna":           {"id": "openai/gpt-5.6-luna",             "cat": "B", "price_in": 0.20,  "price_out": 1.20},
    "claude-sonnet-5":        {"id": "anthropic/claude-sonnet-5",       "cat": "B", "price_in": 2.00,  "price_out": 10.00},
    "gemini-3.7-flash":       {"id": "google/gemini-3.7-flash",         "cat": "B", "price_in": 0.75,  "price_out": 3.75},
    "qwen3-vl-32b":           {"id": "qwen/qwen3-vl-32b-instruct",      "cat": "C", "price_in": 0.104, "price_out": 0.416},
    "qwen3-vl-235b":          {"id": "qwen/qwen3-vl-235b-a22b-instruct","cat": "C", "price_in": 0.20,  "price_out": 0.88},
    "glm-4.6v":               {"id": "z-ai/glm-4.6v",                   "cat": "C", "price_in": 0.30,  "price_out": 0.90},
    "qwen3-vl-8b":            {"id": "qwen/qwen3-vl-8b-instruct",       "cat": "C", "price_in": 0.05,  "price_out": 0.10},
    "qwen3-vl-30b-thinking":  {"id": "qwen/qwen3-vl-30b-a3b-thinking",  "cat": "D", "price_in": 0.15,  "price_out": 1.50},
    "minimax-m3-free":        {"id": "minimax/minimax-m3:free",         "cat": "budget", "price_in": 0.0, "price_out": 0.0},
}

MLBT_AC_MODELS = ["gpt-5.6-sol", "claude-opus-5", "gemini-3.1-pro-preview",
                   "gpt-5.6-luna", "claude-sonnet-5", "gemini-3.7-flash",
                   "qwen3-vl-32b", "qwen3-vl-235b", "glm-4.6v", "qwen3-vl-8b"]
CRG_MODELS = MLBT_AC_MODELS + ["qwen3-vl-30b-thinking"]
JUDGE_MODEL = "claude-opus-5"


## 2. Load and curate the annotation CSV

In [ ]:

df = pd.read_csv(DATA_CSV)
TAG_FAMILIES = ["pedestrian_behavior_tags", "vehicle_tags", "environment_tags", "archetypes"]

def split_tags(cell) -> list[str]:
    if pd.isna(cell) or str(cell).strip() == "":
        return []
    return [t.strip() for t in str(cell).split(",") if t.strip()]

for fam in TAG_FAMILIES:
    df[fam + "_list"] = df[fam].apply(split_tags)

NON_INTERACTION_ONLY_TAGS = set()  # TODO: fill from your curation log

def is_interaction_scenario(row) -> bool:
    all_tags = set(row["pedestrian_behavior_tags_list"]) | set(row["vehicle_tags_list"])
    if not all_tags:
        return False
    if NON_INTERACTION_ONLY_TAGS and all_tags <= NON_INTERACTION_ONLY_TAGS:
        return False
    return True

df_curated = df[df.apply(is_interaction_scenario, axis=1)].reset_index(drop=True)
vocab = {fam: sorted({t for row in df_curated[fam + "_list"] for t in row}) for fam in TAG_FAMILIES}
print(f"{len(df_curated)} curated scenarios; vocab sizes: " +
      ", ".join(f"{fam}={len(v)}" for fam, v in vocab.items()))

ATTENTION_TAGS = [t for t in vocab["pedestrian_behavior_tags"]
                   if any(k in t for k in ["look", "glanc", "fixat", "phone", "head"])]
OUTCOME_TAGS   = [t for t in vocab["pedestrian_behavior_tags"]
                   if any(k in t for k in ["collision", "near-miss", "thrown-back", "recovery"])]


## 3. Evaluation protocol: explicit strata (new in v3)

Because every model here is evaluated zero-shot (no training on this dataset),
"train/test split" doesn't apply in the usual sense — instead these are
**evaluation strata** that let you report performance broken out by difficulty
and by how well-supported an archetype is in the dataset, rather than a single
pooled number. **Important honesty note for the paper's methodology section:**
because these are pretrained, closed-data VLMs rather than models you trained
yourself, a stratum being "unseen" in this CSV does not guarantee the underlying
concept was unseen in the model's pretraining — frame this as *performance
stratification*, not *leakage-controlled generalization*, in the writeup. This
distinction matters to an ICRA reviewer and conflating the two is a common and
easily-caught overclaim.


In [ ]:

def assign_difficulty_level(behavior_tags: list[str]) -> str:
    n = len(set(behavior_tags))
    if n <= 1:
        return "L1_isolated"
    elif n <= 3:
        return "L2_compound"
    else:
        return "L3_complex"

df_curated["difficulty_level"] = df_curated["pedestrian_behavior_tags_list"].apply(assign_difficulty_level)
print(df_curated["difficulty_level"].value_counts())


In [ ]:

def assign_archetype_support_stratum(archetypes: list[str], support_counts: dict, frequent_threshold=100) -> str:
    if not archetypes:
        return "no_archetype"
    max_support = max(support_counts.get(a, 0) for a in archetypes)
    return "frequent" if max_support >= frequent_threshold else "rare"

archetype_support = Counter(a for tags in df_curated["archetypes_list"] for a in set(tags))
df_curated["archetype_support_stratum"] = df_curated["archetypes_list"].apply(
    lambda tags: assign_archetype_support_stratum(tags, archetype_support))
print(df_curated["archetype_support_stratum"].value_counts())
print("\nPer-archetype support (used to threshold frequent/rare):")
print(pd.Series(archetype_support).sort_values(ascending=False))


In [ ]:

# Standard stratum = the full curated set. These two are the additional ones;
# Sec. 16 builds the third (Compose-lite seen/unseen tag-pair stratum).
STRATA = {
    "standard": df_curated,
    "L1_isolated": df_curated[df_curated["difficulty_level"] == "L1_isolated"],
    "L2_compound": df_curated[df_curated["difficulty_level"] == "L2_compound"],
    "L3_complex": df_curated[df_curated["difficulty_level"] == "L3_complex"],
    "archetype_frequent": df_curated[df_curated["archetype_support_stratum"] == "frequent"],
    "archetype_rare": df_curated[df_curated["archetype_support_stratum"] == "rare"],
}
for name, sub in STRATA.items():
    print(f"{name}: {len(sub)} scenarios")


## 4. Dataset analysis — Tables II-V (pure CSV stats, no VLM)

In [ ]:

def archetype_coverage(df, min_signature_frac=0.40, top_n_signature=2):
    rows = []
    for arche in vocab["archetypes"]:
        sub = df[df["archetypes_list"].apply(lambda tags: arche in tags)]
        n = len(sub)
        if n == 0:
            continue
        behavior_counts = Counter(t for tags in sub["pedestrian_behavior_tags_list"] for t in tags)
        signature = [(t, c / n) for t, c in behavior_counts.most_common() if c / n >= min_signature_frac][:top_n_signature]
        rows.append({"archetype": arche.upper(), "scenarios": n,
                      "signature": "; ".join(f"{t} ({frac:.1%})" for t, frac in signature)})
    return pd.DataFrame(rows).sort_values("scenarios", ascending=False).reset_index(drop=True)

table_ii = archetype_coverage(df_curated)
table_ii.to_csv(RESULTS_DIR / "table_ii_archetype_coverage.csv", index=False)
table_ii


In [ ]:

PROFILE_LABELS = ["collision", "near-miss", "run-into-traffic", "ignore-traffic", "looking"]

def archetype_outcome_profile(df, labels=PROFILE_LABELS):
    rows = []
    for arche in vocab["archetypes"]:
        sub = df[df["archetypes_list"].apply(lambda tags: arche in tags)]
        n = len(sub)
        if n == 0:
            continue
        row = {"archetype": arche.upper(), "scenarios": n}
        for label in labels:
            hit = sub["pedestrian_behavior_tags_list"].apply(lambda tags: label in tags).sum()
            row[label] = round(100 * hit / n, 1)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("scenarios", ascending=False).reset_index(drop=True)

table_iii = archetype_outcome_profile(df_curated)
table_iii.to_csv(RESULTS_DIR / "table_iii_behavior_outcome_profile.csv", index=False)
table_iii


In [ ]:

def archetype_overlaps(df, top_k=10):
    counts = {a: df["archetypes_list"].apply(lambda tags: a in tags).sum() for a in vocab["archetypes"]}
    co = Counter()
    for tags in df["archetypes_list"]:
        uniq = sorted(set(tags))
        for i in range(len(uniq)):
            for j in range(i + 1, len(uniq)):
                co[(uniq[i], uniq[j])] += 1
    pairs = [p for p, _ in co.most_common(top_k)]
    rows = []
    for a, b in pairs:
        shared = df.apply(lambda r: (a in r["archetypes_list"]) and (b in r["archetypes_list"]), axis=1).sum()
        rows.append({"archetype_a": a.upper(), "archetype_b": b.upper(), "shared": shared,
                      "within_a_pct": round(100 * shared / counts[a], 1) if counts[a] else 0.0,
                      "within_b_pct": round(100 * shared / counts[b], 1) if counts[b] else 0.0})
    return pd.DataFrame(rows)

table_iv = archetype_overlaps(df_curated)
table_iv.to_csv(RESULTS_DIR / "table_iv_archetype_overlaps.csv", index=False)
table_iv


In [ ]:

def occlusion_subset(df, anchor_tag="pop-out-occlusion"):
    sub = df[df["pedestrian_behavior_tags_list"].apply(lambda tags: anchor_tag in tags)]
    n = len(sub)
    secondary_counts = Counter(t for tags in sub["pedestrian_behavior_tags_list"] for t in tags if t != anchor_tag)
    rows = [{"secondary_label": t, "scenarios": c, "within_subset_pct": round(100 * c / n, 2)}
            for t, c in secondary_counts.most_common()]
    return n, pd.DataFrame(rows)

subset_n, table_v = occlusion_subset(df_curated)
print(f"Occlusion subset size: {subset_n} scenarios")
table_v.to_csv(RESULTS_DIR / "table_v_occlusion_subset.csv", index=False)
table_v.head(10)


## 5. Non-generative baselines (new in v3)

Every VLM number in the paper should be read against these. A model that doesn't
clear the frequency baseline by a wide margin on MLBT isn't demonstrating
recognition — it's mostly reproducing the tag-frequency prior.


In [ ]:

def build_frequency_baseline_tags(train_df, family, k):
    counts = Counter(t for tags in train_df[family + "_list"] for t in tags)
    return [t for t, _ in counts.most_common(k)]

def run_baseline(df, family, mode="frequency", k=None, seed=RANDOM_SEED):
    assert mode in ("frequency", "random")
    gold = df[family + "_list"].apply(lambda l: sorted(set(l))).tolist()
    k = k or max(1, round(np.mean([len(g) for g in gold])))
    if mode == "frequency":
        pred_tags = build_frequency_baseline_tags(df, family, k)
        preds = [pred_tags for _ in range(len(df))]
    else:
        rng = random.Random(seed)
        vocab_list = vocab[family]
        preds = [sorted(rng.sample(vocab_list, min(k, len(vocab_list)))) for _ in range(len(df))]
    return pd.DataFrame({"scenario_id": df["id"].values, "model": f"baseline_{mode}",
                          "gold": gold, "pred": preds})

baseline_freq_mlbt = run_baseline(df_curated, "pedestrian_behavior_tags", mode="frequency")
baseline_rand_mlbt = run_baseline(df_curated, "pedestrian_behavior_tags", mode="random")
baseline_freq_ac = run_baseline(df_curated, "archetypes", mode="frequency")
baseline_rand_ac = run_baseline(df_curated, "archetypes", mode="random")
print("Baseline prediction (frequency, MLBT):", baseline_freq_mlbt["pred"].iloc[0])
print("Baseline prediction (frequency, AC):", baseline_freq_ac["pred"].iloc[0])
# Scored alongside real models with score_multilabel() in Sec. 9/10 by concatenating
# these frames with the model results before calling score_multilabel.


## 6. Video acquisition and frame extraction

In [ ]:

def download_source_video(url: str) -> Path | None:
    vid_id_match = re.search(r"(?:v=|/)([0-9A-Za-z_-]{11})", url)
    vid_id = vid_id_match.group(1) if vid_id_match else re.sub(r"\\W+", "_", url)[-16:]
    out_path = VIDEO_DIR / f"{vid_id}.mp4"
    if out_path.exists():
        return out_path
    result = subprocess.run(["yt-dlp", "-f", "mp4", "-o", str(out_path), url], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[WARN] failed to download {url}: {result.stderr[-300:]}")
        return None
    return out_path

def get_source_fps(video_path: Path) -> float:
    import cv2
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()
    return fps

def extract_scenario_frames(video_path: Path, scenario_id, start_frame: int, end_frame: int) -> list[Path]:
    import cv2
    out_dir = FRAME_DIR / str(scenario_id)
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(out_dir.glob("frame_*.jpg"))
    if existing:
        return existing
    fps = get_source_fps(video_path)
    n = num_frames_for_scenario(start_frame, end_frame, fps)
    frame_indices = np.linspace(start_frame, end_frame, n, dtype=int)
    cap = cv2.VideoCapture(str(video_path))
    saved = []
    for i, fidx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
        ok, frame = cap.read()
        if not ok:
            continue
        fp = out_dir / f"frame_{i:03d}.jpg"
        cv2.imwrite(str(fp), frame)
        saved.append(fp)
    cap.release()
    return saved

def build_frame_index(df, limit=None):
    index = {}
    unique_videos = df["video_path"].unique()
    if limit:
        unique_videos = unique_videos[:limit]
    video_cache = {url: download_source_video(url) for url in tqdm(unique_videos, desc="videos")}
    for _, row in tqdm(df.iterrows(), total=len(df), desc="scenarios"):
        vp = video_cache.get(row["video_path"])
        if vp is None:
            continue
        frames = extract_scenario_frames(vp, row["id"], int(row["start_frame"]), int(row["end_frame"]))
        index[row["id"]] = [str(f) for f in frames]
    return index

# frame_index = build_frame_index(df_curated)          # full run
# frame_index = build_frame_index(df_curated, limit=5)  # smoke test first


## 7. Taxonomy definitions (fill in from your PedAnalyze docs)

In [ ]:

TAXONOMY_DEFINITIONS = {
    fam: {tag: "TODO: paste operational definition from PedAnalyze docs" for tag in tags}
    for fam, tags in vocab.items()
}

def taxonomy_block(fam: str) -> str:
    return "\n".join(f"- {tag}: {TAXONOMY_DEFINITIONS[fam][tag]}" for tag in vocab[fam])


## 8. Unified OpenRouter model wrapper (+ manifest hooks)

Same as v2, with `PROMPT_REGISTRY` added so Sec. 19 can hash and log the exact
prompt text used for every task, not just the model IDs.


In [ ]:

def _encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

CALL_LOG = []
PROMPT_REGISTRY = {}  # name -> prompt text, populated as each task's prompts are defined below

def call_openrouter(model_key: str, system_prompt: str, user_text: str,
                     frame_paths: list[str], max_retries=3, sleep_s=1.5) -> str:
    model_id = MODEL_REGISTRY[model_key]["id"]
    content = [{"type": "text", "text": user_text}]
    for fp in frame_paths:
        content.append({"type": "image_url",
                         "image_url": {"url": f"data:image/jpeg;base64,{_encode_image(fp)}"}})
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model_id, temperature=0, max_tokens=1500,
                messages=[{"role": "system", "content": system_prompt},
                          {"role": "user", "content": content}],
            )
            usage = getattr(resp, "usage", None)
            CALL_LOG.append({
                "model": model_key, "timestamp": datetime.datetime.utcnow().isoformat(),
                "prompt_tokens": getattr(usage, "prompt_tokens", None) if usage else None,
                "completion_tokens": getattr(usage, "completion_tokens", None) if usage else None,
            })
            return resp.choices[0].message.content
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * (attempt + 1))
    raise RuntimeError(f"OpenRouter call failed for {model_key} after {max_retries} attempts: {last_err}")


## 9. Task 1 — MLBT: Multi-Label Behavior Tag Prediction

In [ ]:

MLBT_SYSTEM_PROMPT = (
    "You are annotating dashcam video clips of pedestrian-vehicle interactions using "
    "a fixed behavior taxonomy. Given a sequence of frames from one clip, identify every "
    "applicable pedestrian behavior tag. Respond with ONLY a comma-separated list of tags "
    "from the provided vocabulary — no other text."
)
PROMPT_REGISTRY["MLBT_SYSTEM_PROMPT"] = MLBT_SYSTEM_PROMPT

def mlbt_user_prompt(n_frames: int) -> str:
    return (f"Here are {n_frames} frames sampled uniformly from a pre-event dashcam clip. "
            f"What pedestrian behaviors are observable in this clip? Select all that apply "
            f"from the taxonomy below.\n\nTAXONOMY:\n{taxonomy_block('pedestrian_behavior_tags')}\n\n"
            f"Answer as a comma-separated list of tags only.")

def parse_tag_list(raw_text: str, tag_vocab: list[str], fuzzy_threshold=85) -> list[str]:
    text = raw_text.strip().strip(".")
    text = re.sub(r"^```.*?```$", "", text, flags=re.DOTALL).strip()
    candidates = [t.strip().lower() for t in re.split(r"[,\n]", text) if t.strip()]
    matched = []
    for c in candidates:
        exact = next((v for v in tag_vocab if v.lower() == c), None)
        if exact:
            matched.append(exact)
            continue
        best = rf_process.extractOne(c, tag_vocab, scorer=fuzz.token_sort_ratio)
        if best and best[1] >= fuzzy_threshold:
            matched.append(best[0])
    return sorted(set(matched))

def run_mlbt(df, frame_index, model_keys=MLBT_AC_MODELS):
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="MLBT scenarios"):
        frames = frame_index.get(row["id"])
        if not frames:
            continue
        gold = sorted(set(row["pedestrian_behavior_tags_list"]))
        prompt = mlbt_user_prompt(len(frames))
        for mk in model_keys:
            try:
                raw = call_openrouter(mk, MLBT_SYSTEM_PROMPT, prompt, frames)
                pred = parse_tag_list(raw, vocab["pedestrian_behavior_tags"])
                status = "ok"
            except Exception as e:
                pred, status, raw = [], f"error: {e}", None
            records.append({"scenario_id": row["id"], "model": mk, "status": status,
                             "gold": gold, "pred": pred, "raw": raw})
    return pd.DataFrame(records)

# mlbt_results = run_mlbt(df_curated, frame_index)
# mlbt_results.to_json(RESULTS_DIR / "mlbt_raw.jsonl", orient="records", lines=True)


In [ ]:

def score_multilabel(results_df, tag_vocab, gold_col="gold", pred_col="pred"):
    rows = []
    for model_name, sub in results_df.groupby("model"):
        tp = fp = fn = 0
        for _, r in sub.iterrows():
            g, p = set(r[gold_col]), set(r[pred_col])
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        micro_p = tp / (tp + fp) if (tp + fp) else 0.0
        micro_r = tp / (tp + fn) if (tp + fn) else 0.0
        micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0

        per_tag = []
        for tag in tag_vocab:
            g = sub[gold_col].apply(lambda l: tag in l)
            if g.sum() == 0:
                continue
            p = sub[pred_col].apply(lambda l: tag in l)
            tpv, fpv, fnv = (g & p).sum(), (p & ~g).sum(), (g & ~p).sum()
            pr = tpv / (tpv + fpv) if (tpv + fpv) else 0.0
            rc = tpv / (tpv + fnv) if (tpv + fnv) else 0.0
            f1v = 2 * pr * rc / (pr + rc) if (pr + rc) else 0.0
            per_tag.append({"tag": tag, "precision": pr, "recall": rc, "f1": f1v, "support": int(g.sum())})
        macro_f1 = float(np.mean([t["f1"] for t in per_tag])) if per_tag else float("nan")

        rows.append({"model": model_name, "micro_precision": round(micro_p, 3),
                      "micro_recall": round(micro_r, 3), "micro_f1": round(micro_f1, 3),
                      "macro_f1": round(macro_f1, 3), "n_scenarios": len(sub), "per_tag": per_tag})
    return pd.DataFrame(rows)

# Include baselines alongside real models for the main table:
# combined = pd.concat([mlbt_results[["scenario_id","model","gold","pred"]],
#                        baseline_freq_mlbt, baseline_rand_mlbt], ignore_index=True)
# mlbt_scores = score_multilabel(combined, vocab["pedestrian_behavior_tags"])
# mlbt_scores.drop(columns="per_tag").to_csv(RESULTS_DIR / "table_mlbt_main.csv", index=False)


## 10. Task 2 — AC: Archetype Classification (oracle / pipeline)

In [ ]:

AC_SYSTEM_PROMPT = (
    "You are classifying pedestrian archetypes in dashcam clips of pedestrian-vehicle "
    "interactions, using a fixed archetype taxonomy. Given frames and a set of behavior "
    "tags already identified for this pedestrian, decide which archetype(s) best describe "
    "the behavior pattern. Respond with ONLY a comma-separated list of archetypes from the "
    "provided vocabulary — no other text."
)
PROMPT_REGISTRY["AC_SYSTEM_PROMPT"] = AC_SYSTEM_PROMPT

def ac_user_prompt(n_frames: int, behavior_tags: list[str]) -> str:
    tags_str = ", ".join(behavior_tags) if behavior_tags else "(none identified)"
    return (f"Here are {n_frames} frames from a pre-event dashcam clip. The following pedestrian "
            f"behavior tags have already been identified: {tags_str}\n\n"
            f"Based on this behavior pattern, which archetype(s) best describe the pedestrian? "
            f"Select all that apply.\n\nTAXONOMY:\n{taxonomy_block('archetypes')}\n\n"
            f"Answer as a comma-separated list of archetypes only.")

def run_ac(df, frame_index, mlbt_results=None, model_keys=MLBT_AC_MODELS, condition="oracle"):
    assert condition in ("oracle", "pipeline")
    if condition == "pipeline":
        assert mlbt_results is not None, "pipeline condition needs mlbt_results from Sec. 9"
        pred_lookup = {(r["scenario_id"], r["model"]): r["pred"] for _, r in mlbt_results.iterrows()}

    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"AC ({condition}) scenarios"):
        frames = frame_index.get(row["id"])
        if not frames:
            continue
        gold = sorted(set(row["archetypes_list"]))
        for mk in model_keys:
            behavior_tags = (sorted(set(row["pedestrian_behavior_tags_list"])) if condition == "oracle"
                              else pred_lookup.get((row["id"], mk), []))
            prompt = ac_user_prompt(len(frames), behavior_tags)
            try:
                raw = call_openrouter(mk, AC_SYSTEM_PROMPT, prompt, frames)
                pred = parse_tag_list(raw, vocab["archetypes"])
                status = "ok"
            except Exception as e:
                pred, status, raw = [], f"error: {e}", None
            records.append({"scenario_id": row["id"], "model": mk, "condition": condition,
                             "status": status, "gold": gold, "pred": pred, "raw": raw})
    return pd.DataFrame(records)

# ac_oracle   = run_ac(df_curated, frame_index, condition="oracle")
# ac_pipeline = run_ac(df_curated, frame_index, mlbt_results=mlbt_results, condition="pipeline")
# ac_scores_oracle   = score_multilabel(ac_oracle, vocab["archetypes"])
# ac_scores_pipeline = score_multilabel(ac_pipeline, vocab["archetypes"])
# gap = ac_scores_oracle.set_index("model")["macro_f1"] - ac_scores_pipeline.set_index("model")["macro_f1"]
# gap.rename("error_propagation_gap").to_csv(RESULTS_DIR / "table_ac_error_propagation_gap.csv")


## 11. Task 3 — CRG: Causal Rationale Generation (7 axes, oracle / pipeline)

In [ ]:

CRG_AXES = {
    "BTR": ("Behavioral Trajectory Reasoning",
            "What sequence of micro-behaviors led to the risk event? Describe the temporal "
            "arc of pedestrian actions from clip start to the critical moment. Reference "
            "specific observable transitions (e.g. stationary -> brisk-walk -> run-into-traffic)."),
    "ASR": ("Attentional State Reasoning",
            "Assess the pedestrian's situational awareness. Were they looking? Glancing "
            "without processing? Fully preoccupied (phone, object, other person)? Back-turned? "
            "Blind to the vehicle's approach?"),
    "VPID": ("Vehicle-Pedestrian Interaction Dynamics",
             "Describe the interplay between pedestrian and vehicle behavior. Did the vehicle "
             "brake aggressively? Maintain speed? Was the pedestrian's path predictable or "
             "chaotic from the driver's perspective?"),
    "ERA": ("Environmental Risk Amplification",
            "Which scene-level factors amplified the risk? Night conditions, no crosswalk, "
            "occlusion, one-way vs. two-way road, traffic density, presence of other "
            "pedestrians. Note how many compounding risk factors are present."),
    "AAR": ("Archetype Attribution Reasoning",
            "Justify why the pedestrian fits their assigned archetype(s). Connect behavior to "
            "a causal typology."),
    "CIR": ("Counterfactual Intervention Reasoning",
            "What single change by the pedestrian, vehicle, or environment would most likely "
            "have prevented the risk event? Reason about causal sufficiency, not just "
            "correlation."),
    "OSA": ("Outcome Severity Assessment",
            "Classify the observable outcome: near-miss, collision, thrown-back, "
            "collision-with-recovery, or another category if evident. Justify using visual "
            "evidence only."),
}

CRG_SYSTEM_PROMPT = (
    "You are generating structured causal reasoning about a pedestrian-vehicle safety "
    "scenario from dashcam footage, across seven fixed reasoning axes. Base every claim on "
    "the frames provided and the behavior/archetype tags given to you — do not invent facts "
    "not supported by the visual evidence."
)
PROMPT_REGISTRY["CRG_SYSTEM_PROMPT"] = CRG_SYSTEM_PROMPT

def crg_user_prompt(n_frames: int, behavior_tags: list[str], archetypes: list[str]) -> str:
    axis_block = "\n".join(f"{i+1}. {code} — {name}: {desc}"
                            for i, (code, (name, desc)) in enumerate(CRG_AXES.items()))
    return (f"Here are {n_frames} frames from a pre-event dashcam clip.\n"
            f"Identified behavior tags: {', '.join(behavior_tags) or '(none)'}\n"
            f"Identified archetype(s): {', '.join(archetypes) or '(none)'}\n\n"
            f"Generate structured reasoning across these 7 axes:\n{axis_block}\n\n"
            f"Respond with exactly 7 sections, each starting with the axis code in brackets, "
            f"e.g.:\n[BTR] <paragraph>\n[ASR] <paragraph>\n...\n[OSA] <paragraph>")

def parse_crg_response(raw_text: str) -> dict:
    sections = {}
    pattern = r"\[(" + "|".join(CRG_AXES.keys()) + r")\]\s*(.*?)(?=\[(?:" + "|".join(CRG_AXES.keys()) + r")\]|$)"
    for code, paragraph in re.findall(pattern, raw_text, flags=re.DOTALL):
        sections[code] = paragraph.strip()
    for code in CRG_AXES:
        sections.setdefault(code, "")
    return sections

def run_crg(df, frame_index, mlbt_results=None, ac_results=None,
            model_keys=CRG_MODELS, condition="oracle"):
    assert condition in ("oracle", "pipeline")
    if condition == "pipeline":
        assert mlbt_results is not None and ac_results is not None
        mlbt_lookup = {(r["scenario_id"], r["model"]): r["pred"] for _, r in mlbt_results.iterrows()}
        ac_lookup = {(r["scenario_id"], r["model"]): r["pred"]
                     for _, r in ac_results[ac_results["condition"] == "pipeline"].iterrows()}

    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"CRG ({condition}) scenarios"):
        frames = frame_index.get(row["id"])
        if not frames:
            continue
        for mk in model_keys:
            if condition == "oracle":
                behavior_tags = sorted(set(row["pedestrian_behavior_tags_list"]))
                archetypes = sorted(set(row["archetypes_list"]))
            else:
                behavior_tags = mlbt_lookup.get((row["id"], mk), [])
                archetypes = ac_lookup.get((row["id"], mk), [])
            prompt = crg_user_prompt(len(frames), behavior_tags, archetypes)
            try:
                raw = call_openrouter(mk, CRG_SYSTEM_PROMPT, prompt, frames)
                sections = parse_crg_response(raw)
                status = "ok"
            except Exception as e:
                sections, status, raw = {c: "" for c in CRG_AXES}, f"error: {e}", None
            rec = {"scenario_id": row["id"], "model": mk, "condition": condition, "status": status,
                   "gold_behavior_tags": sorted(set(row["pedestrian_behavior_tags_list"])),
                   "gold_vehicle_tags": sorted(set(row["vehicle_tags_list"])),
                   "gold_environment_tags": sorted(set(row["environment_tags_list"])),
                   "gold_archetypes": sorted(set(row["archetypes_list"]))}
            rec.update({f"text_{c}": sections[c] for c in CRG_AXES})
            records.append(rec)
    return pd.DataFrame(records)

# Run CRG on a stratified subsample first (see Sec. 18's sampler) before the full sweep.
# crg_oracle = run_crg(df_curated.sample(150, random_state=RANDOM_SEED), frame_index, condition="oracle")


In [ ]:

def grounding_precision(text: str, gold_tags: list[str], fuzzy_threshold=80) -> float:
    '''Fraction of gold tags for this axis that are mentioned (exactly or fuzzily) in the text.
    This is a coverage / grounding-recall proxy, not a hallucination check -- pair it with a
    read of a few examples before trusting it as a standalone number.'''
    if not gold_tags:
        return float("nan")
    text_low = text.lower()
    hits = 0
    for tag in gold_tags:
        tag_words = tag.replace("-", " ")
        if tag_words in text_low or fuzz.partial_ratio(tag_words, text_low) >= fuzzy_threshold:
            hits += 1
    return hits / len(gold_tags)

def score_crg_grounding(crg_df):
    rows = []
    for _, r in crg_df.iterrows():
        rows.append({
            "scenario_id": r["scenario_id"], "model": r["model"], "condition": r["condition"],
            "BTR_coverage_proxy": grounding_precision(r["text_BTR"], r["gold_behavior_tags"]),
            "ASR_grounding": grounding_precision(r["text_ASR"], [t for t in r["gold_behavior_tags"] if t in ATTENTION_TAGS]),
            "VPID_grounding": grounding_precision(r["text_VPID"], r["gold_vehicle_tags"]),
            "ERA_grounding": grounding_precision(r["text_ERA"], r["gold_environment_tags"]),
            "AAR_grounding": grounding_precision(r["text_AAR"], r["gold_archetypes"]),
            "CIR_grounding": float("nan"),
            "OSA_grounding": grounding_precision(r["text_OSA"], [t for t in r["gold_behavior_tags"] if t in OUTCOME_TAGS]),
        })
    return pd.DataFrame(rows)

# crg_grounding = score_crg_grounding(crg_oracle)


## 12. CRG — LLM-judge CLAIR-style scoring (0-100 per axis)

In [ ]:

JUDGE_SYSTEM_PROMPT = (
    "You are an expert judge evaluating AI-generated causal reasoning about a "
    "pedestrian-vehicle safety scenario. Score the given paragraph for the specified "
    "reasoning axis on a 0-100 scale, considering: (1) groundedness — does it stick to "
    "claims consistent with the provided evidence rather than inventing detail; "
    "(2) specificity — concrete and precise rather than generic hedge language; "
    "(3) causal soundness — for axes involving cause/effect (BTR, VPID, CIR, AAR), does the "
    "claimed causal link actually make sense. Where no ground truth exists for a claim "
    "(true for the CIR axis), judge plausibility instead of correctness. "
    "Respond with ONLY a JSON object: {\"score\": <0-100 integer>, \"reason\": \"<one sentence>\"}"
)
PROMPT_REGISTRY["JUDGE_SYSTEM_PROMPT"] = JUDGE_SYSTEM_PROMPT

def judge_crg_axis(axis_code: str, paragraph: str, gold_evidence: dict) -> dict:
    name, desc = CRG_AXES[axis_code]
    user_text = (f"Axis: {axis_code} ({name})\nAxis definition: {desc}\n\n"
                 f"Available ground-truth evidence for this scenario: {json.dumps(gold_evidence)}\n\n"
                 f"Model's generated paragraph:\n{paragraph}\n\n"
                 f"Score this paragraph 0-100 for the {axis_code} axis.")
    raw = call_openrouter(JUDGE_MODEL, JUDGE_SYSTEM_PROMPT, user_text, frame_paths=[])
    try:
        obj = json.loads(re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE))
        return {"score": int(obj["score"]), "reason": obj.get("reason", "")}
    except Exception:
        return {"score": None, "reason": f"unparseable judge response: {raw[:200]}"}

def judge_crg_row(row) -> dict:
    evidence_by_axis = {
        "BTR": {"behavior_tags": row["gold_behavior_tags"]},
        "ASR": {"attention_tags": [t for t in row["gold_behavior_tags"] if t in ATTENTION_TAGS]},
        "VPID": {"vehicle_tags": row["gold_vehicle_tags"]},
        "ERA": {"environment_tags": row["gold_environment_tags"]},
        "AAR": {"archetypes": row["gold_archetypes"]},
        "CIR": {"note": "no ground truth for counterfactuals - judge plausibility only"},
        "OSA": {"outcome_tags": [t for t in row["gold_behavior_tags"] if t in OUTCOME_TAGS]},
    }
    result = {"scenario_id": row["scenario_id"], "model": row["model"], "condition": row["condition"]}
    for axis in CRG_AXES:
        j = judge_crg_axis(axis, row[f"text_{axis}"], evidence_by_axis[axis])
        result[f"{axis}_judge_score"] = j["score"]
    return result

# judge_scores = pd.DataFrame([judge_crg_row(r) for _, r in tqdm(crg_oracle.iterrows(), total=len(crg_oracle))])
# judge_scores.to_csv(RESULTS_DIR / "table_crg_judge_scores.csv", index=False)


## 13. Judge validation against human ratings (new in v3)

An LLM judge score is not evidence on its own — an ICRA reviewer will (rightly)
ask how you know the judge is measuring what you claim. This section builds a
human-rating template for a small subsample, then computes Spearman correlation
between judge and human scores per axis. Report this correlation table in the
paper alongside the judge scores; axes with weak correlation should be presented
as descriptive/exploratory rather than as a validated metric.


In [ ]:

def build_human_rating_template(crg_df, n_scenarios=50, models_subset=None, seed=RANDOM_SEED, out_path=None):
    rng = np.random.default_rng(seed)
    sample = crg_df.sample(min(n_scenarios, len(crg_df)), random_state=seed)
    if models_subset:
        sample = sample[sample["model"].isin(models_subset)]
    rows = []
    for _, r in sample.iterrows():
        for axis in CRG_AXES:
            rows.append({"scenario_id": r["scenario_id"], "model": r["model"], "axis": axis,
                         "paragraph": r[f"text_{axis}"], "human_score_0_100": ""})  # rater fills this column
    template = pd.DataFrame(rows)
    out_path = out_path or (RESULTS_DIR / "human_rating_template.csv")
    template.to_csv(out_path, index=False)
    print(f"Wrote {len(template)} rows ({len(template)//len(CRG_AXES)} scenario-model pairs) to {out_path}")
    print("Have a human rater fill the human_score_0_100 column, then reload with load_human_ratings().")
    return template

def load_human_ratings(path=None) -> pd.DataFrame:
    path = path or (RESULTS_DIR / "human_rating_template.csv")
    ratings = pd.read_csv(path)
    ratings = ratings[ratings["human_score_0_100"] != ""].copy()
    ratings["human_score_0_100"] = ratings["human_score_0_100"].astype(float)
    return ratings

def judge_vs_human_correlation(judge_scores_df, human_ratings_df) -> pd.DataFrame:
    rows = []
    for axis in CRG_AXES:
        merged = human_ratings_df[human_ratings_df["axis"] == axis].merge(
            judge_scores_df[["scenario_id", "model", f"{axis}_judge_score"]],
            on=["scenario_id", "model"], how="inner")
        merged = merged.dropna(subset=["human_score_0_100", f"{axis}_judge_score"])
        if len(merged) < 5:
            rows.append({"axis": axis, "n": len(merged), "spearman_r": float("nan"), "p_value": float("nan")})
            continue
        r, p = sstats.spearmanr(merged["human_score_0_100"], merged[f"{axis}_judge_score"])
        rows.append({"axis": axis, "n": len(merged), "spearman_r": round(r, 3), "p_value": round(p, 4)})
    return pd.DataFrame(rows)

# build_human_rating_template(crg_oracle, n_scenarios=50, models_subset=["gpt-5.6-sol", "qwen3-vl-32b"])
# ... send the CSV to a rater, get it back, then:
# human_ratings = load_human_ratings()
# corr_table = judge_vs_human_correlation(judge_scores, human_ratings)
# corr_table.to_csv(RESULTS_DIR / "table_judge_human_correlation.csv", index=False)
# Rule of thumb for the paper: Spearman r >= 0.5 -> report the axis as judge-validated;
# below that, report it as descriptive only (this will very plausibly be the outcome for CIR).


## 14. Statistical significance (new in v3)

Bootstrap confidence intervals for every micro-F1 number, plus paired bootstrap
significance tests between the top models on each task. Report CIs in every main
table; report significance stars only for comparisons you actually care about
(e.g. best open-weight vs. best frontier model) rather than every pair, to avoid
a multiple-comparisons mess.


In [ ]:

def bootstrap_ci_micro_f1(results_df, gold_col="gold", pred_col="pred", n_boot=2000, seed=RANDOM_SEED, ci=0.95):
    per_scenario = np.array([
        (len(set(r[gold_col]) & set(r[pred_col])), len(set(r[pred_col]) - set(r[gold_col])),
         len(set(r[gold_col]) - set(r[pred_col])))
        for _, r in results_df.iterrows()
    ])
    n = len(per_scenario)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    sums = per_scenario[idx].sum(axis=1)  # (n_boot, 3)
    tp, fp, fn = sums[:, 0], sums[:, 1], sums[:, 2]
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp, dtype=float), where=(prec + rec) > 0)
    lo, hi = np.percentile(f1, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return float(f1.mean()), float(lo), float(hi)

def score_with_ci(results_df, model_col="model", n_boot=2000, seed=RANDOM_SEED):
    rows = []
    for model_name, sub in results_df.groupby(model_col):
        mean_f1, lo, hi = bootstrap_ci_micro_f1(sub, n_boot=n_boot, seed=seed)
        rows.append({"model": model_name, "micro_f1": round(mean_f1, 3),
                      "ci_lo": round(lo, 3), "ci_hi": round(hi, 3), "n_scenarios": len(sub)})
    return pd.DataFrame(rows).sort_values("micro_f1", ascending=False)

# mlbt_ci_table = score_with_ci(mlbt_results)
# mlbt_ci_table.to_csv(RESULTS_DIR / "table_mlbt_with_ci.csv", index=False)


In [ ]:

def paired_bootstrap_test(results_df, model_a: str, model_b: str, n_boot=2000, seed=RANDOM_SEED):
    '''Paired bootstrap test on micro-F1 difference between two models, resampling the
    same scenario indices for both so the pairing is preserved. Returns the mean delta,
    a 95% CI on the delta, and a two-sided p-value approximation from the bootstrap
    distribution's sign.'''
    a = results_df[results_df["model"] == model_a].set_index("scenario_id")
    b = results_df[results_df["model"] == model_b].set_index("scenario_id")
    common_ids = a.index.intersection(b.index)
    a, b = a.loc[common_ids], b.loc[common_ids]

    def per_scenario_tfpn(df):
        return np.array([(len(set(g) & set(p)), len(set(p) - set(g)), len(set(g) - set(p)))
                          for g, p in zip(df["gold"], df["pred"])])

    arr_a, arr_b = per_scenario_tfpn(a), per_scenario_tfpn(b)
    n = len(arr_a)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))

    def boot_f1(arr):
        sums = arr[idx].sum(axis=1)
        tp, fp, fn = sums[:, 0], sums[:, 1], sums[:, 2]
        prec = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
        rec  = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0)
        return np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp, dtype=float), where=(prec + rec) > 0)

    f1_a, f1_b = boot_f1(arr_a), boot_f1(arr_b)
    delta = f1_a - f1_b
    lo, hi = np.percentile(delta, [2.5, 97.5])
    p_approx = 2 * min((delta <= 0).mean(), (delta >= 0).mean())
    return {"model_a": model_a, "model_b": model_b, "mean_delta_f1": round(float(delta.mean()), 4),
            "ci_lo": round(float(lo), 4), "ci_hi": round(float(hi), 4),
            "p_approx": round(float(p_approx), 4), "significant_at_0.05": not (lo < 0 < hi)}

# paired_bootstrap_test(mlbt_results, "gpt-5.6-sol", "qwen3-vl-32b")


## 15. Stress tests — DOPe-Compose-lite and DOPe-Early-lite (with CIs)

In [ ]:

def build_seen_unseen_split(df, family="pedestrian_behavior_tags_list", n_holdout_pairs=15, seed=RANDOM_SEED):
    rng = random.Random(seed)
    all_pairs = set()
    for tags in df[family]:
        uniq = sorted(set(tags))
        for i in range(len(uniq)):
            for j in range(i + 1, len(uniq)):
                all_pairs.add((uniq[i], uniq[j]))
    holdout_pairs = set(rng.sample(sorted(all_pairs), min(n_holdout_pairs, len(all_pairs))))

    def contains_holdout(tags):
        uniq = sorted(set(tags))
        return any((uniq[i], uniq[j]) in holdout_pairs
                   for i in range(len(uniq)) for j in range(i + 1, len(uniq)))

    is_unseen = df[family].apply(contains_holdout)
    return df[~is_unseen].reset_index(drop=True), df[is_unseen].reset_index(drop=True), holdout_pairs

seen_split, unseen_split, holdout_pairs = build_seen_unseen_split(df_curated)
STRATA["compose_seen"] = seen_split
STRATA["compose_unseen"] = unseen_split
print(f"seen: {len(seen_split)}, unseen: {len(unseen_split)}, held-out pairs (sample): {list(holdout_pairs)[:5]}")

# mlbt_seen   = run_mlbt(seen_split, frame_index)
# mlbt_unseen = run_mlbt(unseen_split, frame_index)
# seen_ci, unseen_ci = score_with_ci(mlbt_seen), score_with_ci(mlbt_unseen)
# delta_comp = seen_ci.set_index("model")["micro_f1"] - unseen_ci.set_index("model")["micro_f1"]
# Report delta_comp alongside the CIs from both strata, not as a bare point estimate.


In [ ]:

TRUNCATION_FRACTIONS = [1.0, 0.75, 0.5, 0.25]

def build_truncated_frame_index(df, frame_index, fractions=TRUNCATION_FRACTIONS):
    truncated = {frac: {} for frac in fractions}
    for _, row in df.iterrows():
        frames = frame_index.get(row["id"])
        if not frames:
            continue
        for frac in fractions:
            keep_n = max(1, round(len(frames) * frac))
            truncated[frac][row["id"]] = frames[:keep_n]
    return truncated

# truncated_index = build_truncated_frame_index(df_curated, frame_index)
# for frac, idx in truncated_index.items():
#     res = run_mlbt(df_curated, idx)
#     res["truncation_fraction"] = frac
#     score_with_ci(res).assign(truncation_fraction=frac).to_csv(
#         RESULTS_DIR / f"early_lite_mlbt_frac_{frac}.csv", index=False)
# Plot micro_f1 (with CI band) vs truncation_fraction per model as the context-sufficiency curve.


## 16. Failure taxonomy (F1-F7)

In [ ]:

def flag_failure_types(mlbt_row, ac_row):
    '''Coarse triage flags -- for manual qualitative review, not a substitute for it.'''
    flags = []
    gold_beh, pred_beh = set(mlbt_row["gold"]), set(mlbt_row["pred"])
    gold_arc, pred_arc = set(ac_row["gold"]), set(ac_row["pred"])

    if gold_beh - pred_beh:
        flags.append("F1_behavior_miss")
    if not (gold_beh - pred_beh) and gold_arc != pred_arc and gold_arc and pred_arc:
        flags.append("F5_composition_failure_candidate")
    if "collision" in gold_beh and "collision" not in pred_beh:
        flags.append("F6_outcome_confusion")
    return flags

# Join mlbt_results and ac_oracle on (scenario_id, model) and apply row-wise.


## 17. Cost & throughput estimator

In [ ]:

def estimate_run_cost(n_scenarios: int, avg_frames_per_scenario: float, model_keys: list[str],
                       avg_output_tokens: int, tokens_per_frame: int = 800) -> pd.DataFrame:
    '''Rough estimate -- image tokenization varies by provider, so calibrate
    tokens_per_frame against Sec. 8 CALL_LOG after a ~20-scenario pilot and re-run.'''
    rows = []
    for mk in model_keys:
        m = MODEL_REGISTRY[mk]
        input_tokens_per_call = avg_frames_per_scenario * tokens_per_frame + 300
        cost_per_call = (input_tokens_per_call / 1e6) * m["price_in"] + (avg_output_tokens / 1e6) * m["price_out"]
        rows.append({"model": mk, "cost_per_call_usd": round(cost_per_call, 4),
                      "total_calls": n_scenarios, "total_cost_usd": round(cost_per_call * n_scenarios, 2)})
    return pd.DataFrame(rows).sort_values("total_cost_usd", ascending=False)

print("MLBT full sweep (910 scenarios, ~8 frames avg):")
print(estimate_run_cost(910, 8, MLBT_AC_MODELS, avg_output_tokens=60).to_string(index=False))
print("\\nCRG on a 150-scenario pilot (7 paragraphs, ~900 output tokens):")
print(estimate_run_cost(150, 8, CRG_MODELS, avg_output_tokens=900).to_string(index=False))
print("(add ~7x the judge model's own cost per scenario for Sec. 12's per-axis judging,")
print(" plus the Sec. 13 human-validation subsample -- unpaid if you rate it yourself)")


## 18. Reproducibility manifest (new in v3)

Write one of these alongside every results file. This is what lets the paper say
"evaluated between DATE and DATE using exactly these model snapshots and
prompts" instead of an unfalsifiable claim about a fixed benchmark artifact.


In [ ]:

def build_run_manifest(strata_used: list[str] = None) -> dict:
    return {
        "run_timestamp_utc": datetime.datetime.utcnow().isoformat(),
        "random_seed": RANDOM_SEED,
        "min_frames": MIN_FRAMES, "sample_fps": SAMPLE_FPS,
        "model_registry": {k: v["id"] for k, v in MODEL_REGISTRY.items()},
        "prompt_hashes": {name: hashlib.sha256(text.encode()).hexdigest()[:12]
                          for name, text in PROMPT_REGISTRY.items()},
        "strata_used": strata_used or list(STRATA.keys()),
        "python_version": platform.python_version(),
        "n_scenarios_total": len(df_curated),
    }

manifest = build_run_manifest()
with open(RESULTS_DIR / "run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
manifest


## 19. Optional appendix — local Qwen3-VL-32B (OpenRouter-independent check)

Unchanged from v2. Not required for the main experimental matrix. GPU sizing:
bf16 needs ~66GB (1x A100/H100-80GB); 4-bit needs ~20-22GB (1x A100-40GB /
L40S-48GB) at a typical 2-5 point F1 cost.


In [ ]:

MODAL_APP_STUB = '''
# modal_app.py — deploy with: modal deploy modal_app.py
import modal

app = modal.App("dope-qwen3-vl-local")
image = (modal.Image.debian_slim(python_version="3.11")
         .pip_install("torch", "transformers>=4.45", "accelerate", "qwen-vl-utils", "pillow", "fastapi"))
MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"

@app.cls(gpu="A100-80GB", image=image, timeout=600, scaledown_window=300)
class Qwen3VL:
    @modal.enter()
    def load(self):
        import torch
        from transformers import AutoModelForVision2Seq, AutoProcessor
        self.model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
        self.processor = AutoProcessor.from_pretrained(MODEL_ID)

    @modal.method()
    def generate(self, system_prompt: str, user_text: str, frames_b64: list[str]) -> str:
        import base64, io
        from PIL import Image
        images = [Image.open(io.BytesIO(base64.b64decode(b))) for b in frames_b64]
        messages = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": [{"type": "text", "text": user_text}] + [{"type": "image"} for _ in images]}]
        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=prompt, images=images, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs, max_new_tokens=512, do_sample=False)
        return self.processor.decode(out[0], skip_special_tokens=True)

@app.function(image=image)
@modal.fastapi_endpoint(method="POST")
def endpoint(payload: dict):
    return {"text": Qwen3VL().generate.remote(payload["system_prompt"], payload["user_text"], payload["frames_b64"])}
'''
with open("modal_app_local_qwen.py", "w") as f:
    f.write(MODAL_APP_STUB)
print("Wrote modal_app_local_qwen.py")


## 20. Limitations

- **Table I (inter-annotator kappa)**: needs the two raw pre-adjudication
  PedAnalyze exports, not this merged CSV.
- **True DOPe-Risk / DOPe-Intervene**: not attempted — need TTC/distance/velocity
  and counterfactual (agent, action, timing) annotation respectively.
- **DOPe-Early-lite**: a context-truncation proxy, not a lead-time measurement.
- **CRG-BTR**: grounding score is tag-coverage, not sequence validity.
- **CRG-CIR**: no ground truth; judge-scored for plausibility only.
- **Evaluation strata are not leakage-controlled** (Sec. 3): pretrained VLMs may
  have prior exposure to similar concepts; report strata as performance
  breakdowns, not generalization guarantees.
- **Judge reliability**: validate against human ratings (Sec. 13) before
  reporting CRG judge scores as a primary metric; report per-axis correlation,
  not an assumed-reliable single number.
- **Model drift**: OpenRouter-served models can be silently deprecated/replaced
  mid-project (Sec. 0). Pin and report the manifest (Sec. 18) alongside every
  results table, and state the evaluation window explicitly in the paper.
